# Notebook_D (dùng chung, phần mô hình, Sinh viên B): chia dữ liệu, sáu họ mô hình, tổng quát hoá, tin cậy, giải thích và chẩn đoán

**Dùng chung cho mọi đề tài.** Nhận `data/processed/feat.parquet` và `manifest.json` từ Notebook_C, đọc CONFIG như A, chạy tuần tự. Trả lời **RQ2** (mô hình nào tốt nhất theo tầm và có tổng quát hoá theo thời gian, theo đơn vị không) và **RQ3** (bao phủ khoảng tin cậy, cảnh báo sự kiện, đổi chế độ, đóng góp của nguồn phụ), rồi giải thích bằng SHAP và chẩn đoán.

**Sản phẩm**: `report/table_rq2_*.csv`, `report/table_rq3_*.csv`, `report/fig_rq2_*.png`, `report/fig_rq3_*.png`, `report/table_shap.csv`, `report/table_diagnosis.csv`, AI Audit Log cập nhật.

In [1]:
# CONFIG: fill in once for your group, both notebooks read the same block
TOPIC = "Cross-plant Transfer and Conformal Uncertainty for Hourly Solar Power Forecasting on NREL Integration Data"
GROUP = "Group 7"                    # group label used in file names and the audit log
PRIMARY_SOURCE = {"name": "NREL/NLR Solar Power Data for Integration Studies (California)", "url": "https://www.nrel.gov/grid/solar-power-data", "license": "Public access (specific license unconfirmed)", "path": "data/raw/primary.csv"}
SECOND_SOURCE  = {"name": "NSRDB (National Solar Radiation Database) weather data", "url": "https://nsrdb.nlr.gov", "license": "CC BY 4.0", "path": "data/raw/secondary.csv"}
UNIT_COL   = "plant"                  # column that identifies a series (station, site, zone, vm, ...)
TIME_COL   = "ts"                    # timestamp column (will be parsed to datetime)
TARGET_COL = "cf"                     # variable to forecast
EXOG_COLS  = ["GHI", "DNI", "cloud", "temp"]      # columns from the secondary source used as covariates
FREQ       = "h"                     # pandas offset alias of the regular grid: 'h', 'D', 'W', '15min', 'min'
HORIZONS   = [1, 6]                 # forecast horizons in steps of FREQ (e.g. 1 h and 24 h ahead)
SEASON     = 24                      # seasonal period in steps (24 for hourly-daily, 7 for daily-weekly, 52 for weekly-yearly)
TEST_START = "2006-10-01"            # first timestamp of the test period (time-based split)
EVENT_QUANTILE = 0.9                 # "event" = target above this quantile (used by RQ3 warning metrics)
SEEDS = [0, 1, 2, 3, 4]
print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Cross-plant Transfer and Conformal Uncertainty for Hourly Solar Power Forecasting on NREL Integration Data
Group: Group 7


In [2]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Hướng dẫn hỏi AI (điền đề tài của nhóm vào chỗ trống)

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)` ở ô trên. Mẫu prompt theo bước (thay `{TOPIC}`, `{cột}` bằng thông tin thật của nhóm):

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Mô hình | "Đề tài {TOPIC}, mục tiêu {TARGET_COL} tần suất {FREQ}, tầm {HORIZONS}. Với sáu họ mô hình (naive mùa, ridge, random forest, LightGBM toàn cục, tuyến tính dài hạn, mô hình nền zero-shot) hãy nêu siêu tham số nào đáng tinh chỉnh nhất và vì sao." | Chỉ tinh chỉnh khi chênh lệch giữa mô hình lớn hơn chênh lệch giữa seed |
| Thu thập | "Đề tài của tôi là {TOPIC}. Nguồn chính là {PRIMARY_SOURCE}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết, sự kiện, giá, lịch) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép." | Mở URL, xác nhận có tải được và giấy phép cho phép dùng |
| So sánh nguồn | "Tôi có hai nguồn về cùng biến {TARGET_COL} ở tần suất {FREQ}. Hãy đề xuất 4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn." | Chạy lại số trên dữ liệu thật, không dùng số AI đưa |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng giá trị tương lai |
| SQL | "Viết truy vấn DuckDB tạo đặc trưng trễ {SEASON} bước và trung bình trượt theo {UNIT_COL}, có WINDOW, không rò rỉ." | Đếm dòng, kiểm tra cột NULL ở đầu chuỗi |
| EDA | "Từ bảng thống kê sau (dán bảng), nêu 3 nhận xét có thể kiểm chứng và 2 điều cần cảnh giác." | Mỗi nhận xét phải chỉ ra được ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại ô test |

Ví dụ ghi Audit Log sau khi hỏi: `audit("Thu thập", "Đề tài ... gợi ý nguồn phụ", "Claude", "3 nguồn: NOAA ISD, ...", "mở 3 URL, 1 URL lỗi 404", "dùng NOAA ISD, loại nguồn 404", hallucination=True)`.

## Bước 0: nhận bàn giao và kiểm tra

**Làm gì**: đọc manifest, so mã MD5, nạp bảng đặc trưng vào DuckDB. **Vì sao**: nếu A sửa dữ liệu sau khi bàn giao mà B không biết, kết quả hai người sẽ không khớp; MD5 phát hiện ngay.

In [3]:
# Step 0: imports, style, handover check
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, time, pickle
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
man = json.load(open("data/processed/manifest.json"))
assert hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest() == man["md5"], "parquet differs from the manifest: ask Student A for the current file"
UNIT_COL, TIME_COL, TARGET_COL, FREQ, HORIZONS, SEASON, TEST_START = man["unit_col"], man["time_col"], man["target_col"], man["freq"], man["horizons"], man["season"], man["test_start"]
EXOG_COLS = man["exog_cols"]
feat = pd.read_parquet("data/processed/feat.parquet"); feat[TIME_COL] = pd.to_datetime(feat[TIME_COL])
# alternative when only the CSV was shared: feat = pd.read_csv("data/processed/feat.csv", parse_dates=[TIME_COL])
con = duckdb.connect(); con.register("feat", feat)
print("handover OK:", man["rows"], "rows,", man["units"], "units,", man["start"], "to", man["end"])

handover OK: 109141 rows, 20 units, 2006-01-01 07:00:00 to 2006-12-31 17:00:00


## Bước 4: chia dữ liệu và mốc naive

**Ba cách chia, mỗi cách trả lời một câu hỏi**: (1) **theo thời gian** (huấn luyện trước `TEST_START`, kiểm thử sau) trả lời "mô hình có dùng được cho tương lai không"; (2) **lùi gốc dự báo** (rolling origin, nhiều điểm cắt) trả lời "kết quả có ổn định theo thời gian không"; (3) **để dành đơn vị** (leave-units-out) trả lời "mô hình có dùng cho trạm, vùng, khách hàng mới không".

**Mốc naive** là điều kiện tối thiểu: mô hình nào không thắng mốc naive thì không đáng báo cáo. Hai mốc: persistence (giá trị hiện tại) và naive mùa (giá trị cách đúng một mùa).

In [4]:
# Step 4a: feature list and the primary horizon; drop rows whose lags or targets are missing (start of each unit, end of each unit)
H0 = HORIZONS[0]; Y = f"y_h{H0}"
lag_cols = [c for c in feat.columns if c.startswith("y_lag") or c.startswith("y_ma") or c.startswith("y_sd") or c == "y_diff1"]
exog_now = [c for c in EXOG_COLS if c in feat.columns]; exog_lag = [c for c in feat.columns if c.endswith("_lag_season")]
cal_cols = [c for c in ["hr", "dow", "mon", "doy"] if c in feat.columns]
feat["unit_code"] = feat[UNIT_COL].astype("category").cat.codes
X_cols = [TARGET_COL] + lag_cols + exog_now + exog_lag + cal_cols + ["unit_code"]
data = feat.dropna(subset=X_cols + [f"y_h{h}" for h in HORIZONS]).copy()
print("features:", X_cols); print("rows usable:", len(data), "of", len(feat))

features: ['cf', 'y_lag1', 'y_lag2', 'y_lag3', 'y_lag24', 'y_lag48', 'y_lag168', 'y_ma_season', 'y_sd_season', 'y_diff1', 'GHI', 'DNI', 'cloud', 'temp', 'GHI_lag_season', 'DNI_lag_season', 'cloud_lag_season', 'temp_lag_season', 'hr', 'dow', 'mon', 'doy', 'unit_code']
rows usable: 77905 of 109141


In [5]:
# Step 4b: time-based split
train = data[data[TIME_COL] < pd.Timestamp(TEST_START)]; test = data[data[TIME_COL] >= pd.Timestamp(TEST_START)]
X_tr, y_tr, X_te, y_te = train[X_cols], train[Y], test[X_cols], test[Y]
assert len(train) > 0 and len(test) > 0, "check TEST_START"
print(f"train {len(train):,} rows to {train[TIME_COL].max().date()} | test {len(test):,} rows from {test[TIME_COL].min().date()}")

train 60,498 rows to 2006-09-30 | test 17,407 rows from 2006-10-01


In [6]:
# Step 4c: naive baselines for every horizon (persistence = current value; seasonal naive = value one season before the target time)
def naive_preds(frame, h):
    pers = frame[TARGET_COL].values
    seas = frame[f"y_lag{SEASON}"].values if h <= SEASON and f"y_lag{SEASON}" in frame.columns else frame[TARGET_COL].values   # seasonal naive valid when h <= SEASON
    return pers, seas
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE
for h in HORIZONS:
    pers, seas = naive_preds(test, h); yt = test[f"y_h{h}"].values
    print(f"h={h}: persistence MAE {MAE(yt, pers):.4g} | seasonal naive MAE {MAE(yt, seas):.4g}")

h=1: persistence MAE 0.1146 | seasonal naive MAE 0.2219
h=6: persistence MAE 0.4118 | seasonal naive MAE 0.2797


## Chỉ số đánh giá và cách đọc

| Chỉ số | Công thức ngắn | Khi nào dùng | Cạm bẫy |
|---|---|---|---|
| MAE | trung bình \|y - ŷ\| | mọi bài, dễ hiểu, cùng đơn vị với mục tiêu | không phạt nặng sai số lớn |
| RMSE | căn trung bình (y - ŷ)² | khi sai số lớn tốn kém hơn | bị chi phối bởi ngoại lai |
| MAPE | trung bình \|y - ŷ\| / \|y\| | so sánh giữa đơn vị có quy mô khác nhau | nổ khi y gần 0; không dùng cho biến có số 0 |
| sMAPE | 2\|y - ŷ\| / (\|y\| + \|ŷ\|) | thay MAPE khi y có thể bằng 0 | không đối xứng thật sự |
| MASE | MAE / MAE của naive trong tập huấn luyện | so sánh giữa đề tài, giữa tầm; dưới 1 là thắng naive | cần naive cùng mùa |
| Pinball, coverage | phạt phân vị; tỷ lệ y nằm trong khoảng | dự báo xác suất và khoảng tin cậy | bao phủ đúng nhưng khoảng quá rộng là vô dụng, luôn báo cả độ rộng |

Trong bài: bảng chính báo MAE và MASE (kèm độ lệch chuẩn qua seed); bảng phụ báo RMSE, sMAPE; RQ3 báo coverage và độ rộng.

In [7]:
# Metrics helper (MASE scaled by the in-sample seasonal naive MAE of each unit)
def smape(y, p): return float(np.mean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p) + 1e-9)) * 100)
def mase(y, p, frame, h):
    scale = np.mean(np.abs(train[f"y_h{h}"].values - naive_preds(train, h)[1])) + 1e-9
    return float(np.mean(np.abs(y - p)) / scale)
def metrics(y, p, frame, h):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": MAE(y, p), "RMSE": MSE(y, p) ** 0.5, "sMAPE": smape(y, p), "MASE": mase(y, p, frame, h)}
print(metrics(y_te.values, naive_preds(test, H0)[1], test, H0))

{'MAE': 0.22188192067415843, 'RMSE': 0.290725740765865, 'sMAPE': 78.8725400973674, 'MASE': 0.5359066628597965}


## Bước 6a: sáu họ mô hình, năm seed, mọi tầm dự báo

**Họ mô hình và lý do chọn**: naive mùa (điều kiện tối thiểu); ridge với trễ (tuyến tính, nhanh, khó overfit); random forest (phi tuyến, ít siêu tham số); LightGBM toàn cục (mạnh nhất cho bảng đặc trưng, học chung nhiều đơn vị); tuyến tính dài hạn (LTSF-Linear: một lớp tuyến tính từ cửa sổ trễ, mốc mạnh của các bài mô hình nền); mô hình nền zero-shot (Chronos, TimesFM: không huấn luyện, dự báo từ ngữ cảnh, đại diện cho xu hướng 2024-2025).

**Năm seed**: mô hình có ngẫu nhiên (random forest, LightGBM) chạy 5 lần; báo trung bình và độ lệch chuẩn. Nếu chênh lệch giữa hai mô hình nhỏ hơn hai lần độ lệch chuẩn thì không kết luận mô hình nào hơn.

In [8]:
# Step 6a-1: model zoo (direct strategy: one model per horizon, features at time t predict y at t+h)
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
def make_models(seed):
    return {"Ridge": Ridge(alpha=1.0),
            "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=seed),
            "LightGBM_global": LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=seed, verbose=-1),
            "LTSF_Linear": Ridge(alpha=0.1)}          # LTSF-Linear reduces to a linear map from the lag window; we feed only lag columns below
LTSF_COLS = [c for c in lag_cols if c.startswith("y_lag")] + [TARGET_COL]
print(list(make_models(0).keys()))

['Ridge', 'RandomForest', 'LightGBM_global', 'LTSF_Linear']


In [9]:
# Step 6a-2: run all models x seeds x horizons; store per-row predictions for the primary horizon
results, preds = [], {}
for h in HORIZONS:
    ytr, yte = train[f"y_h{h}"].values, test[f"y_h{h}"].values
    pers, seas = naive_preds(test, h)
    for name, p in [("Persistence", pers), ("SeasonalNaive", seas)]:
        m = metrics(yte, p, test, h); results.append({"model": name, "h": h, "seed": 0, **m})
        if h == H0: preds[name] = p
    for seed in SEEDS:
        for name, model in make_models(seed).items():
            cols = LTSF_COLS if name == "LTSF_Linear" else X_cols
            model.fit(train[cols], ytr); p = model.predict(test[cols])
            results.append({"model": name, "h": h, "seed": seed, **metrics(yte, p, test, h)})
            if h == H0 and seed == 0: preds[name] = p
            if name in ("Ridge", "LTSF_Linear"): break               # deterministic: one seed is enough
res = pd.DataFrame(results); res.to_csv("report/table_rq2_raw.csv", index=False)
summary = res.groupby(["model", "h"])[["MAE", "RMSE", "sMAPE", "MASE"]].agg(["mean", "std"]).round(4); summary.to_csv("report/table_rq2_summary.csv"); display(summary)

MAE         RMSE          sMAPE         MASE     
                   mean  std    mean  std      mean  std    mean  std
model         h                                                      
Persistence   1  0.1146  NaN  0.1531  NaN   49.5413  NaN  0.2768  NaN
              6  0.4118  NaN  0.4732  NaN  129.1494  NaN  1.4294  NaN
Ridge         1  0.0746  0.0  0.0987  0.0   44.6920  0.0  0.1802  0.0
              6  0.1869  0.0  0.2278  0.0   85.8585  0.0  0.6488  0.0
SeasonalNaive 1  0.2219  NaN  0.2907  NaN   78.8725  NaN  0.5359  NaN
              6  0.2797  NaN  0.3511  NaN  109.1142  NaN  0.9708  NaN

In [10]:
# Step 6a-3: zero-shot foundation models (Chronos; TimesFM optional). Skipped automatically if the package is not installed.
fm_rows = []
try:
    from chronos import ChronosPipeline; import torch
    pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="cpu")
    CTX = min(512, 8 * SEASON); n_units = min(20, test[UNIT_COL].nunique()); yt_all, yp_all = [], []
    for u in test[UNIT_COL].unique()[:n_units]:
        hist = data[(data[UNIT_COL] == u) & (data[TIME_COL] < pd.Timestamp(TEST_START))][TARGET_COL].values[-CTX:]
        fut = test[test[UNIT_COL] == u].sort_values(TIME_COL)[TARGET_COL].values
        # rolling zero-shot: forecast H0 steps, then slide the context by H0 (cheap approximation of an operational loop)
        ctx = list(hist); k = 0
        while k + H0 <= len(fut):
            fc = pipe.predict(torch.tensor(np.array(ctx[-CTX:], dtype=float)), prediction_length=H0).median(dim=1).values.numpy().ravel()
            yp_all.extend(fc); yt_all.extend(fut[k:k + H0]); ctx.extend(fut[k:k + H0]); k += H0
            if k > 40 * H0: break                                     # limit runtime: 40 windows per unit
    yt_all, yp_all = np.array(yt_all), np.array(yp_all)
    fm_rows.append({"model": "Chronos_zero_shot", "h": H0, "seed": 0, **metrics(yt_all, yp_all, test, H0)})
    print("Chronos evaluated on", len(yt_all), "steps")
except Exception as e:
    print("foundation model skipped:", str(e)[:120], "-> report as a limitation or run on Colab with pip install chronos-forecasting")
if fm_rows: res = pd.concat([res, pd.DataFrame(fm_rows)]); res.to_csv("report/table_rq2_raw.csv", index=False)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 7416.53it/s]


Chronos evaluated on 820 steps


In [11]:
# Step 6a-4: main comparison figure (bold numbers on bars, no error bars, no in-figure title) and paired significance test
main = res[res.h == H0].groupby("model").MAE.agg(["mean", "std"]).sort_values("mean")
fig, ax = plt.subplots(figsize=(8, 3.4)); bars = ax.bar(main.index, main["mean"], color=[ACC if i == 0 else TEAL for i in range(len(main))])
ax.bar_label(bars, fmt="%.3g", fontweight="bold", fontsize=9, padding=2); ax.set_ylabel(f"MAE, horizon {H0}"); ax.set_ylim(0, main["mean"].max() * 1.2); plt.setp(ax.get_xticklabels(), rotation=15); style(ax); savefig(fig, "fig_rq2_models")
from scipy.stats import wilcoxon
best_name = main.index[0]; runner = main.index[1]
if best_name in preds and runner in preds:
    e1, e2 = np.abs(y_te.values - preds[best_name]), np.abs(y_te.values - preds[runner])
    print(f"Wilcoxon {best_name} vs {runner}: p = {wilcoxon(e1, e2).pvalue:.2e}; report as significant only if p < 0.01 and the gap exceeds 2 seed standard deviations")

saved report/fig_rq2_models.png


In [12]:
# Step 6a-5: error by horizon (does the ranking change with the lead time?)
byh = res.groupby(["model", "h"]).MAE.mean().unstack(0)
fig, ax = plt.subplots(figsize=(7, 3.3))
for m in byh.columns: ax.plot(byh.index, byh[m], marker="o", label=m)
ax.set_xlabel("horizon (steps)"); ax.set_ylabel("MAE"); ax.legend(frameon=False, fontsize=8, ncol=2); style(ax); savefig(fig, "fig_rq2_horizon")
byh.round(4).to_csv("report/table_rq2_by_horizon.csv"); display(byh.round(4))

saved report/fig_rq2_horizon.png


model,Chronos_zero_shot,Persistence,Ridge,SeasonalNaive
h,,,,
1,0.0927,0.1146,0.0746,0.2219
6,NaN,0.4118,0.1869,0.2797


In [13]:
# Step 6a-6: best model object for the rest of the notebook (retrained with seed 0 on the primary horizon)
best_name = [m for m in main.index if m in make_models(0)][0]
best = make_models(0)[best_name]; best_cols = LTSF_COLS if best_name == "LTSF_Linear" else X_cols
best.fit(train[best_cols], y_tr); pred_best = best.predict(test[best_cols])
print("best trainable model:", best_name, "MAE", round(MAE(y_te, pred_best), 4))

best trainable model: Ridge MAE 0.0746


### Tinh chỉnh siêu tham số: chỉ khi đáng

**Quy tắc**: tinh chỉnh chỉ có ý nghĩa nếu chênh lệch giữa mô hình tốt nhất và mô hình kế tiếp lớn hơn hai lần độ lệch chuẩn theo seed; nếu không thì thời gian nên dành cho đặc trưng và dữ liệu. Dùng `TimeSeriesSplit` để không rò rỉ tương lai trong tinh chỉnh.

In [14]:
# Step 6a-7: randomized search on the best model with time-series cross-validation, only if worth it
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
gap = main["mean"].iloc[1] - main["mean"].iloc[0]; sd = res[(res.h == H0) & (res.model == main.index[0])].MAE.std(); sd = 0.0 if sd != sd else sd
if best_name == "LightGBM_global" and gap > 2 * (sd if sd == sd else 0):
    grid = {"num_leaves": [15, 31, 63], "learning_rate": [0.01, 0.03, 0.1], "n_estimators": [400, 800, 1500], "min_child_samples": [10, 20, 50]}
    rs = RandomizedSearchCV(LGBMRegressor(random_state=0, verbose=-1), grid, n_iter=8, cv=TimeSeriesSplit(4), scoring="neg_mean_absolute_error", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    print("best params", rs.best_params_, "| tuned MAE", round(MAE(y_te, rs.best_estimator_.predict(X_te)), 4), "vs default", round(MAE(y_te, pred_best), 4))
    if MAE(y_te, rs.best_estimator_.predict(X_te)) < MAE(y_te, pred_best): best = rs.best_estimator_; pred_best = best.predict(X_te)
else:
    print("tuning skipped:", "best model is not LightGBM" if best_name != "LightGBM_global" else f"model gap {gap:.4g} is within seed noise {2 * sd:.4g}")

tuning skipped: best model is not LightGBM


In [15]:
# Step 6a-8: actual versus predicted for one unit in the test period, and error by calendar period
u0 = test[UNIT_COL].unique()[0]; g = test[test[UNIT_COL] == u0].assign(pred=pred_best[test[UNIT_COL].values == u0]).sort_values(TIME_COL).head(6 * SEASON)
fig, ax = plt.subplots(figsize=(11, 3.2)); ax.plot(g[TIME_COL], g[Y], color=GREY, label="actual"); ax.plot(g[TIME_COL], g.pred, color=TEAL, label=best_name); ax.legend(frameon=False); ax.set_ylabel(Y); style(ax); savefig(fig, "fig_rq2_actual_vs_pred")
err = test.assign(err=np.abs(y_te.values - pred_best))
fig, axes = plt.subplots(1, len(cal_cols[:3]), figsize=(11, 3.0))
for ax, k in zip(np.atleast_1d(axes), cal_cols[:3]):
    s = err.groupby(k).err.mean(); ax.bar(s.index, s.values, color=TEAL); ax.set_xlabel(k); ax.set_ylabel("MAE"); style(ax)
savefig(fig, "fig_rq2_error_by_calendar")

saved report/fig_rq2_actual_vs_pred.png
saved report/fig_rq2_error_by_calendar.png


## RQ2 sâu hơn: tổng quát hoá theo thời gian và theo đơn vị

**Lùi gốc dự báo (rolling origin)**: huấn luyện đến điểm cắt k, kiểm thử một cửa sổ sau đó, lặp 4-6 lần. Nếu MAE dao động mạnh giữa các cửa sổ, kết quả phụ thuộc giai đoạn và bài phải báo khoảng. **Để dành đơn vị**: giữ 20% đơn vị hoàn toàn ngoài huấn luyện; tổn thất so với đơn vị đã thấy cho biết mô hình có "học chung" hay chỉ nhớ từng đơn vị.

In [16]:
# RQ2-1: rolling-origin backtest with the best model family (5 cut points evenly spaced in the test period)
cuts = pd.date_range(pd.Timestamp(TEST_START), data[TIME_COL].max(), periods=7)[1:-1]
rows = []
for c in cuts:
    tr_r = data[data[TIME_COL] < c]; te_r = data[(data[TIME_COL] >= c) & (data[TIME_COL] < c + (cuts[1] - cuts[0]))]
    if len(te_r) < 100: continue
    m_r = make_models(0)[best_name].fit(tr_r[best_cols], tr_r[Y]); p_r = m_r.predict(te_r[best_cols])
    rows.append([c.date(), len(te_r), MAE(te_r[Y], p_r), MAE(te_r[Y], naive_preds(te_r, H0)[1])])
roll = pd.DataFrame(rows, columns=["cut", "rows", "MAE_model", "MAE_seasonal_naive"]).round(4); roll.to_csv("report/table_rq2_rolling.csv", index=False); display(roll)
fig, ax = plt.subplots(figsize=(7, 3.2)); ax.plot(roll.cut.astype(str), roll.MAE_model, marker="o", color=TEAL, label=best_name); ax.plot(roll.cut.astype(str), roll.MAE_seasonal_naive, marker="s", color=ACC, label="seasonal naive"); ax.legend(frameon=False); ax.set_ylabel("MAE"); plt.setp(ax.get_xticklabels(), rotation=20); style(ax); savefig(fig, "fig_rq2_rolling")

,cut,rows,MAE_model,MAE_seasonal_naive
0,2006-10-16,3061,0.0605,0.2608
1,2006-10-31,2954,0.0657,0.2568
2,2006-11-15,2720,0.0699,0.1755
3,2006-11-30,2769,0.0647,0.1297
4,2006-12-16,2844,0.0803,0.1865


saved report/fig_rq2_rolling.png


In [17]:
# RQ2-2: leave-units-out: hold out 20% of units entirely, compare seen versus unseen units and the naive baseline
units = data[UNIT_COL].unique(); rs = np.random.RandomState(42); hold = rs.choice(units, max(1, len(units) // 5), replace=False)
tr_u = train[~train[UNIT_COL].isin(hold)]; te_seen = test[~test[UNIT_COL].isin(hold)]; te_unseen = test[test[UNIT_COL].isin(hold)]
cols_u = [c for c in best_cols if c != "unit_code"]                    # unit_code is meaningless for unseen units
m_u = make_models(0)[best_name].fit(tr_u[cols_u], tr_u[Y])
lou = pd.DataFrame([["seen units", MAE(te_seen[Y], m_u.predict(te_seen[cols_u])), MAE(te_seen[Y], naive_preds(te_seen, H0)[1])],
                    ["unseen units", MAE(te_unseen[Y], m_u.predict(te_unseen[cols_u])), MAE(te_unseen[Y], naive_preds(te_unseen, H0)[1])]], columns=["group", "MAE_model", "MAE_seasonal_naive"]).round(4)
lou.to_csv("report/table_rq2_unseen_units.csv", index=False); display(lou)
print("relative loss on unseen units:", round(lou.MAE_model.iloc[1] / lou.MAE_model.iloc[0] - 1, 3))

,group,MAE_model,MAE_seasonal_naive
0,seen units,0.0725,0.2182
1,unseen units,0.0826,0.2368


relative loss on unseen units: 0.139


In [18]:
# RQ2-3: per-unit results (which units are hard?) for the best model
pu = test.assign(err=np.abs(y_te.values - pred_best)).groupby(UNIT_COL).agg(MAE=("err", "mean"), mean_target=(Y, "mean"), n=("err", "size"))
pu["MAE_rel"] = pu.MAE / pu.mean_target.abs().clip(lower=1e-9); pu = pu.sort_values("MAE_rel"); pu.round(4).to_csv("report/table_rq2_per_unit.csv")
fig, ax = plt.subplots(figsize=(9, 3.2)); ax.bar(range(len(pu)), pu.MAE_rel, color=TEAL); ax.set_xlabel("units sorted by relative MAE"); ax.set_ylabel("MAE / mean target"); style(ax); savefig(fig, "fig_rq2_per_unit")
print("hardest units:", list(pu.index[-3:]))

saved report/fig_rq2_per_unit.png
hardest units: ['Actual_38.65_-121.25_2006_DPV_', 'Actual_38.65_-121.05_2006_DPV_', 'Actual_37.75_-122.05_2006_UPV_']


## Bước 6b (RQ3): tin cậy và vận hành

Bốn phân tích, mỗi phân tích một bảng và một hình:
1. **Khoảng tin cậy conformal 90%**: bảo đảm bao phủ không cần giả định phân phối; báo bao phủ và độ rộng theo đơn vị.
2. **Cảnh báo sự kiện**: sự kiện là mục tiêu vượt phân vị `EVENT_QUANTILE`; quét ngưỡng dự báo, báo recall và precision; thêm ngưỡng conformal trên xác suất để cố định tỷ lệ báo động giả.
3. **Đổi chế độ**: sai số theo tháng trong kỳ kiểm thử; nếu tăng đột ngột thì thử huấn luyện lại cuốn chiếu.
4. **Ablation nguồn phụ**: bỏ toàn bộ biến từ nguồn phụ, đo MAE tăng bao nhiêu; đây là bằng chứng cho từ "multimodal" trong tiêu đề.

In [19]:
# RQ3-1: split conformal intervals (90%) with a calibration slice at the end of training; coverage and width by unit
cal_start = train[TIME_COL].quantile(0.8)
tr_c = train[train[TIME_COL] < cal_start]; cal = train[train[TIME_COL] >= cal_start]
m_c = make_models(0)[best_name].fit(tr_c[best_cols], tr_c[Y])
q = np.quantile(np.abs(cal[Y].values - m_c.predict(cal[best_cols])), 0.9 * (1 + 1 / len(cal)))
p_c = m_c.predict(test[best_cols]); lo, hi = p_c - q, p_c + q
cov = ((y_te.values >= lo) & (y_te.values <= hi))
cov_unit = pd.DataFrame({"unit": test[UNIT_COL].values, "covered": cov}).groupby("unit").covered.mean().round(3)
print("overall coverage", round(cov.mean(), 3), "| half-width", round(q, 4), "| units with coverage < 0.85:", int((cov_unit < 0.85).sum()))
cov_unit.to_csv("report/table_rq3_coverage_by_unit.csv")
fig, ax = plt.subplots(figsize=(9, 3.0)); ax.bar(range(len(cov_unit)), cov_unit.values, color=TEAL); ax.axhline(0.9, color=ACC, ls="--"); ax.set_ylabel("coverage"); ax.set_xlabel("units"); ax.set_ylim(0.5, 1.0); style(ax); savefig(fig, "fig_rq3_coverage")

overall coverage 0.683 | half-width 0.1234 | units with coverage < 0.85: 20
saved report/fig_rq3_coverage.png


In [20]:
# RQ3-2: optional MAPIE (cross-conformal) if installed; usually tighter intervals than split conformal
try:
    from mapie.regression import MapieRegressor
    mp = MapieRegressor(make_models(0)[best_name], method="plus", cv=5).fit(X_tr if best_name != "LTSF_Linear" else train[LTSF_COLS], y_tr)
    pm, itv = mp.predict(X_te if best_name != "LTSF_Linear" else test[LTSF_COLS], alpha=0.1)
    print("MAPIE coverage", round(((y_te.values >= itv[:, 0, 0]) & (y_te.values <= itv[:, 1, 0])).mean(), 3), "| mean width", round((itv[:, 1, 0] - itv[:, 0, 0]).mean(), 4))
except Exception as e: print("MAPIE skipped:", str(e)[:80])

MAPIE skipped: cannot import name 'MapieRegressor' from 'mapie.regression' (c:\Users\rqp\AppDat


In [21]:
# RQ3-3: event warning from the regression forecast: sweep thresholds and report recall, precision, F1
thr_event = train[Y].quantile(EVENT_QUANTILE); truth = y_te.values > thr_event
rows = []
for f in [0.7, 0.8, 0.9, 1.0]:
    alarm = pred_best > f * thr_event; tp = (alarm & truth).sum()
    rows.append([f, round(tp / max(truth.sum(), 1), 3), round(tp / max(alarm.sum(), 1), 3), round(alarm.mean(), 3)])
warn = pd.DataFrame(rows, columns=["threshold_factor", "recall", "precision", "alarm_rate"]); warn["F1"] = (2 * warn.recall * warn.precision / (warn.recall + warn.precision + 1e-9)).round(3)
warn.to_csv("report/table_rq3_warning.csv", index=False); display(warn)
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.plot(warn.threshold_factor, warn.recall, marker="o", color=TEAL, label="recall"); ax.plot(warn.threshold_factor, warn.precision, marker="s", color=ACC, label="precision"); ax.set_xlabel("alarm threshold (x event quantile)"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_rq3_warning")

,threshold_factor,recall,precision,alarm_rate,F1
0,0.7,0.993,0.137,0.390,0.241
1,0.8,0.977,0.204,0.258,0.338
2,0.9,0.838,0.402,0.112,0.543
3,1.0,0.295,0.721,0.022,0.419


saved report/fig_rq3_warning.png


In [22]:
# RQ3-4: event classifier with a conformal false-alarm target (2%): guaranteed alarm rate on normal steps
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score
yb_tr = (y_tr.values > thr_event).astype(int); yb_te = truth.astype(int)
if yb_tr.sum() > 20 and yb_te.sum() > 5:
    clf = LGBMClassifier(n_estimators=400, learning_rate=0.05, class_weight="balanced", random_state=0, verbose=-1).fit(X_tr, yb_tr)
    p_ev = clf.predict_proba(X_te)[:, 1]; q_fa = np.quantile(clf.predict_proba(X_tr[yb_tr == 0])[:, 1], 0.98); alarm = p_ev > q_fa
    print("event PR-AUC", round(average_precision_score(yb_te, p_ev), 3), "| false alarm rate", round(alarm[yb_te == 0].mean(), 4), "| recall", round(alarm[yb_te == 1].mean(), 3))
else: print("too few events for a classifier; report the regression-based warning only")

event PR-AUC 0.788 | false alarm rate 0.0063 | recall 0.549


In [23]:
# RQ3-5: regime shift: error by month in the test period, and monthly retraining if the error jumps
mon = test.assign(err=np.abs(y_te.values - pred_best), naive_err=np.abs(y_te.values - naive_preds(test, H0)[1])).set_index(TIME_COL).resample("MS")[["err", "naive_err"]].mean()
jump = (mon.err / mon.err.iloc[0]).max() if len(mon) and mon.err.iloc[0] > 0 else 1
print("max monthly MAE relative to first test month:", round(float(jump), 2))
rows = []
if len(mon) >= 3:
    for m_start in mon.index[1:]:
        tr_r = data[data[TIME_COL] < m_start]; te_r = data[(data[TIME_COL] >= m_start) & (data[TIME_COL] < m_start + pd.offsets.MonthBegin(1))]
        if len(te_r) == 0: continue
        m_r = make_models(0)[best_name].fit(tr_r[best_cols], tr_r[Y]); rows.append([m_start.date(), MAE(te_r[Y], m_r.predict(te_r[best_cols])), float(mon.loc[m_start, "err"]), float(mon.loc[m_start, "naive_err"])])
regime = pd.DataFrame(rows, columns=["month", "MAE_retrained_monthly", "MAE_fixed_model", "MAE_seasonal_naive"]).round(4); regime.to_csv("report/table_rq3_regime.csv", index=False); display(regime)
if len(regime):
    fig, ax = plt.subplots(figsize=(8, 3.2))
    for c, col in [("MAE_fixed_model", GREY), ("MAE_retrained_monthly", TEAL), ("MAE_seasonal_naive", ACC)]: ax.plot(regime.month.astype(str), regime[c], marker="o", color=col, label=c.replace("MAE_", "").replace("_", " "))
    ax.legend(frameon=False); ax.set_ylabel("MAE"); plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8); style(ax); savefig(fig, "fig_rq3_regime")

max monthly MAE relative to first test month: 1.06


,month,MAE_retrained_monthly,MAE_fixed_model,MAE_seasonal_naive
0,2006-11-01,0.0679,0.0741,0.2164
1,2006-12-01,0.0727,0.0773,0.1584


saved report/fig_rq3_regime.png


In [24]:
# RQ3-6: ablation of the secondary modality (drop all covariate columns) and of calendar features
def fit_eval(cols):
    m = make_models(0)[best_name].fit(train[cols], y_tr); return MAE(y_te, m.predict(test[cols]))
abl = [["full", MAE(y_te, pred_best)]]
if exog_now + exog_lag: abl.append(["without secondary source", fit_eval([c for c in best_cols if c not in exog_now + exog_lag])])
if cal_cols: abl.append(["without calendar", fit_eval([c for c in best_cols if c not in cal_cols])])
abl.append(["lags only", fit_eval([c for c in best_cols if c in lag_cols + [TARGET_COL]])])
abl = pd.DataFrame(abl, columns=["feature_set", "MAE"]).round(4); abl["relative_to_full"] = (abl.MAE / abl.MAE.iloc[0]).round(3); abl.to_csv("report/table_rq3_ablation.csv", index=False); display(abl)
fig, ax = plt.subplots(figsize=(6, 3.2)); bars = ax.bar(abl.feature_set, abl.MAE, color=[ACC] + [TEAL] * (len(abl) - 1)); ax.bar_label(bars, fmt="%.3g", fontweight="bold", fontsize=9); ax.set_ylabel("MAE"); ax.set_ylim(0, abl.MAE.max() * 1.2); plt.setp(ax.get_xticklabels(), rotation=10); style(ax); savefig(fig, "fig_rq3_ablation")

,feature_set,MAE,relative_to_full
0,full,0.0746,1.000
1,without secondary source,0.0709,0.950
2,without calendar,0.0758,1.016
3,lags only,0.0787,1.055


saved report/fig_rq3_ablation.png


In [25]:
# RQ3-7: generative multi-path reference (Chronos samples) for one unit: spread versus conformal interval; skipped if not installed
try:
    from chronos import ChronosPipeline; import torch
    pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="cpu")
    u0 = test[UNIT_COL].unique()[0]; hist = data[(data[UNIT_COL] == u0) & (data[TIME_COL] < pd.Timestamp(TEST_START))][TARGET_COL].values[-min(512, 8 * SEASON):]
    fut = test[test[UNIT_COL] == u0].sort_values(TIME_COL)[TARGET_COL].values[:SEASON]
    samples = pipe.predict(torch.tensor(np.array(hist, dtype=float)), prediction_length=len(fut), num_samples=100).numpy()[0]
    lo_g, med, hi_g = np.quantile(samples, [0.05, 0.5, 0.95], axis=0)
    print("generative paths: coverage 90%", round(((fut >= lo_g) & (fut <= hi_g)).mean(), 3), "| MAE of median path", round(MAE(fut, med), 4))
    fig, ax = plt.subplots(figsize=(9, 3.2)); x = np.arange(len(fut))
    for s in samples[:30]: ax.plot(x, s, color=GREY, lw=0.4, alpha=0.5)
    ax.plot(x, fut, color="k", lw=1.2, label="actual"); ax.plot(x, med, color=TEAL, lw=1.5, label="median of 100 paths"); ax.fill_between(x, lo_g, hi_g, color=TEAL, alpha=0.15); ax.legend(frameon=False); ax.set_xlabel("steps ahead"); style(ax); savefig(fig, "fig_rq3_generative")
except Exception as e: print("generative reference skipped:", str(e)[:80])

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 7631.09it/s]


generative paths: coverage 90% 0.75 | MAE of median path 0.0978
saved report/fig_rq3_generative.png


## Bước 6c: giải thích bằng SHAP và ổn định theo seed

**Đọc SHAP**: thanh dài là đặc trưng chi phối; nếu đặc trưng nguồn phụ nằm trong top 5 thì ablation ở trên nên cho thấy MAE tăng khi bỏ, hai bằng chứng phải nhất quán. **Ổn định**: chạy SHAP với 3 seed, đếm top-5 chung; nếu dưới 3 thì báo tầm quan trọng theo khoảng chứ không theo thứ hạng.

In [26]:
# SHAP importance (tree models) or coefficients (linear), dependence plots for the top three features
imp = None
try:
    import shap
    if best_name in ("LightGBM_global", "RandomForest"):
        ex = shap.TreeExplainer(best); Xs = test[best_cols].sample(min(3000, len(test)), random_state=0); sv = ex.shap_values(Xs)
        imp = pd.DataFrame({"feature": best_cols, "mean_abs_shap": np.abs(sv).mean(0)}).sort_values("mean_abs_shap", ascending=False)
        for f in imp.feature.head(3): shap.dependence_plot(f, sv, Xs, show=False); plt.savefig(f"report/fig_shap_dep_{f}.png", dpi=300, bbox_inches="tight"); plt.close()
    else:
        imp = pd.DataFrame({"feature": best_cols, "mean_abs_shap": np.abs(best.coef_) * test[best_cols].std().values}).sort_values("mean_abs_shap", ascending=False)
except Exception as e: print("SHAP skipped:", str(e)[:80])
if imp is not None:
    imp["share"] = (imp.mean_abs_shap / imp.mean_abs_shap.sum()).round(3); imp.to_csv("report/table_shap.csv", index=False); display(imp.head(10))
    top = imp.head(8)[::-1]; fig, ax = plt.subplots(figsize=(7, 3.4)); bars = ax.barh(top.feature, top.share, color=TEAL); ax.bar_label(bars, fmt="%.2f", fontweight="bold", fontsize=8); ax.set_xlabel("share of mean |SHAP|"); style(ax); savefig(fig, "fig_shap")

,feature,mean_abs_shap,share
0,cf,0.143785,0.277
9,y_diff1,0.060257,0.116
10,GHI,0.055996,0.108
3,y_lag3,0.041812,0.081
1,y_lag1,0.041691,0.080
18,hr,0.037052,0.071
7,y_ma_season,0.023533,0.045
11,DNI,0.021926,0.042
13,temp,0.017099,0.033
12,cloud,0.015241,0.029


saved report/fig_shap.png


In [27]:
# SHAP stability across three seeds (tree models only)
if best_name in ("LightGBM_global", "RandomForest"):
    import shap
    tops = []
    for s in range(3):
        m = make_models(s)[best_name].fit(train[best_cols], y_tr); Xs = test[best_cols].sample(min(2000, len(test)), random_state=s)
        v = shap.TreeExplainer(m).shap_values(Xs); tops.append(set(pd.Series(np.abs(v).mean(0), index=best_cols).nlargest(5).index))
    print("top-5 features shared by all three seeds:", len(tops[0] & tops[1] & tops[2]), "of 5 ->", sorted(tops[0] & tops[1] & tops[2]))

## Chẩn đoán và hướng cải thiện (viết vào Discussion)

| Câu hỏi | Bằng chứng trong notebook | Nếu có vấn đề |
|---|---|---|
| Mô hình có thắng naive mùa không? | Bảng RQ2 summary, MASE dưới 1 | nếu không: đặc trưng thiếu, dùng log1p, hoặc dữ liệu quá nhiễu |
| Kết quả có ổn định theo thời gian? | Bảng rolling: chênh lệch giữa cửa sổ | nếu dao động: báo khoảng, kiểm tra đổi chế độ |
| Có dùng cho đơn vị mới không? | Bảng unseen units | nếu mất trên 30%: thêm đặc trưng mô tả đơn vị |
| Khoảng tin cậy có đúng bao phủ? | Bảng coverage theo đơn vị | đơn vị dưới 0,85: tái hiệu chỉnh riêng |
| Cảnh báo có dùng được? | Bảng warning: F1 ở ngưỡng tốt nhất | precision thấp: nâng ngưỡng hoặc gộp theo cụm |
| Nguồn phụ có đáng không? | Bảng ablation | tăng dưới 2%: cân nhắc bỏ để mô hình nhẹ |
| Giải thích có nhất quán? | SHAP top-5 và ablation | mâu thuẫn: kiểm tra rò rỉ và đa cộng tuyến |

In [28]:
# Diagnosis table assembled from the saved outputs (one row per question, with the number that answers it)
best_row = res[(res.h == H0) & (res.model == best_name)].MASE.mean()
diag = pd.DataFrame([
    ["beats seasonal naive (MASE < 1)", round(best_row, 3)],
    ["rolling-origin spread (max/min MAE)", round(float(roll.MAE_model.max() / max(roll.MAE_model.min(), 1e-9)), 3) if len(roll) else "n/a"],
    ["relative loss on unseen units", round(lou.MAE_model.iloc[1] / lou.MAE_model.iloc[0] - 1, 3)],
    ["conformal coverage (target 0.90)", round(float(cov.mean()), 3)],
    ["best warning F1", float(warn.F1.max())],
    ["MAE increase without secondary source", round(float(abl.set_index("feature_set").relative_to_full.get("without secondary source", np.nan)) - 1, 3) if "without secondary source" in abl.feature_set.values else "n/a"],
], columns=["question", "value"]); diag.to_csv("report/table_diagnosis.csv", index=False); display(diag)

,question,value
0,beats seasonal naive (MASE < 1),0.180
1,rolling-origin spread (max/min MAE),1.327
2,relative loss on unseen units,0.139
3,conformal coverage (target 0.90),0.683
4,best warning F1,0.543
5,MAE increase without secondary source,-0.050


In [29]:
# Test cases for Notebook_D
assert test[TIME_COL].min() > train[TIME_COL].max(), "test must come after train"
assert set(hold).isdisjoint(set(tr_u[UNIT_COL])), "held-out units leaked into training"
assert best_row < 1.05, "best model does not beat the seasonal naive baseline: revisit features before writing results"
for f in ["report/table_rq2_summary.csv", "report/table_rq2_rolling.csv", "report/table_rq2_unseen_units.csv", "report/table_rq3_coverage_by_unit.csv", "report/table_rq3_warning.csv", "report/table_rq3_ablation.csv", "report/table_diagnosis.csv"]:
    assert os.path.exists(f), f"missing {f}"
import os; print("Tests D: OK")

Tests D: OK


In [30]:
# Save the best model and record this week's prompts in the AI Audit Log
pickle.dump({"model": best, "cols": best_cols, "name": best_name}, open("report/best_model.pkl", "wb"))
audit("Step 6a", f"Đề tài {TOPIC}: sáu họ mô hình cho {TARGET_COL} tầm {H0}, siêu tham số nào đáng chỉnh?", "Claude", "khuyên num_leaves và learning_rate", "so chênh lệch mô hình với 2 sd theo seed", "chỉ tinh chỉnh khi gap > 2 sd")
audit("RQ3", "Cách đặt ngưỡng conformal để báo động giả 2%", "Claude", "lấy phân vị 98 của điểm trên bước bình thường trong tập huấn luyện", "đo báo động giả thật trên tập kiểm thử", "áp dụng, báo cả recall")
print(pd.read_csv(AUDIT_PATH).tail(3))

audit entry saved: Step 6a
audit entry saved: RQ3
         date    group     step  \
5  2026-09-22  Group 7      RQ3   
6  2026-09-22  Group 7  Step 6a   
7  2026-09-22  Group 7      RQ3   

                                              prompt    tool  \
5       Cách đặt ngưỡng conformal để báo động giả 2%  Claude   
6  Đề tài Cross-plant Transfer and Conformal Unce...  Claude   
7       Cách đặt ngưỡng conformal để báo động giả 2%  Claude   

                                           ai_output  \
5  lấy phân vị 98 của điểm trên bước bình thường ...   
6                 khuyên num_leaves và learning_rate   
7  lấy phân vị 98 của điểm trên bước bình thường ...   

                               verified_how                       decision  \
5    đo báo động giả thật trên tập kiểm thử         áp dụng, báo cả recall   
6  so chênh lệch mô hình với 2 sd theo seed  chỉ tinh chỉnh khi gap > 2 sd   
7    đo báo động giả thật trên tập kiểm thử         áp dụng, báo cả recall   

   hallucinati

## Bài tập mở rộng cho Sinh viên B (làm ít nhất 3 trong 6, ghi kết quả vào report/exercises_D.md)

1. **Thêm hai họ mô hình**: ExtraTrees và CatBoost (hoặc XGBoost); đưa vào cùng bảng 5 seed; kết luận có đổi thứ hạng không.
2. **Trực tiếp và đệ quy**: với tầm dài nhất, so chiến lược trực tiếp (một mô hình mỗi tầm) và đệ quy (dự báo 1 bước rồi đưa lại làm đầu vào); giải thích vì sao khác nhau.
3. **Hồi quy phân vị**: LightGBM với `objective="quantile"` cho phân vị 0,05 và 0,95; so bao phủ và độ rộng với conformal.
4. **Mô hình từng đơn vị so toàn cục**: huấn luyện một LightGBM cho 10 đơn vị riêng lẻ; so MAE và thời gian với mô hình toàn cục.
5. **Biểu đồ hiệu chuẩn**: với bộ phân loại sự kiện, vẽ reliability diagram và tính ECE; thử temperature scaling.
6. **Ngưỡng theo chi phí**: giả sử bỏ sót sự kiện tốn gấp 5 lần báo động giả; tìm ngưỡng tối thiểu hoá chi phí và so với ngưỡng F1.

In [31]:
# Exercise 1 starter: two more model families in the same protocol
from sklearn.ensemble import ExtraTreesRegressor
extra_rows = []
for seed in SEEDS:
    m = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=seed).fit(X_tr, y_tr)
    extra_rows.append({"model": "ExtraTrees", "h": H0, "seed": seed, **metrics(y_te.values, m.predict(X_te), test, H0)})
print(pd.DataFrame(extra_rows).groupby("model")[["MAE", "MASE"]].agg(["mean", "std"]).round(4))

               MAE            MASE        
              mean     std    mean     std
model                                     
ExtraTrees  0.0583  0.0001  0.1408  0.0002


In [32]:
# Exercise 3 starter: quantile regression intervals with LightGBM (5% and 95%)
q_lo = LGBMRegressor(objective="quantile", alpha=0.05, n_estimators=600, learning_rate=0.03, random_state=0, verbose=-1).fit(X_tr, y_tr).predict(X_te)
q_hi = LGBMRegressor(objective="quantile", alpha=0.95, n_estimators=600, learning_rate=0.03, random_state=0, verbose=-1).fit(X_tr, y_tr).predict(X_te)
print("quantile-regression coverage", round(((y_te.values >= q_lo) & (y_te.values <= q_hi)).mean(), 3), "| mean width", round(float(np.mean(q_hi - q_lo)), 4), "| conformal width", round(2 * q, 4))

quantile-regression coverage 0.828 | mean width 0.2204 | conformal width 0.2467


In [33]:
# Exercise 4 starter: per-unit models versus the global model (10 units)
rows = []
for u in test[UNIT_COL].unique()[:10]:
    trk, tek = train[train[UNIT_COL] == u], test[test[UNIT_COL] == u]
    if len(trk) < 5 * SEASON or len(tek) == 0: continue
    t0 = time.perf_counter(); mk = LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=0, verbose=-1).fit(trk[best_cols], trk[Y]); dt_fit = time.perf_counter() - t0
    rows.append([u, MAE(tek[Y], mk.predict(tek[best_cols])), MAE(tek[Y], pred_best[test[UNIT_COL].values == u]), round(dt_fit, 2)])
print(pd.DataFrame(rows, columns=["unit", "MAE_per_unit_model", "MAE_global_model", "fit_seconds"]).round(4))

                             unit  MAE_per_unit_model  MAE_global_model  \
0  Actual_32.75_-115.45_2006_UPV_              0.0548            0.0837   
1  Actual_33.05_-116.85_2006_DPV_              0.0608            0.0716   
2  Actual_33.25_-114.85_2006_UPV_              0.0468            0.0679   
3  Actual_33.65_-117.25_2006_DPV_              0.0514            0.0609   
4  Actual_33.75_-116.15_2006_UPV_              0.0613            0.0988   
5  Actual_33.75_-116.25_2006_DPV_              0.0462            0.0620   
6  Actual_34.05_-117.15_2006_DPV_              0.0581            0.0667   
7  Actual_34.15_-117.25_2006_DPV_              0.0561            0.0624   
8  Actual_34.25_-117.65_2006_UPV_              0.0617            0.0741   
9  Actual_34.45_-114.65_2006_UPV_              0.0514            0.0689   

   fit_seconds  
0         0.26  
1         0.34  
2         0.34  
3         0.29  
4         0.30  
5         0.29  
6         0.30  
7         0.28  
8         0.29  
9   

In [34]:
# Exercise 5 starter: reliability diagram and ECE for the event classifier (runs only if the classifier exists)
from sklearn.calibration import calibration_curve
if "p_ev" in dir():
    frac, mean_p = calibration_curve(yb_te, p_ev, n_bins=10)
    bins = np.minimum((p_ev * 10).astype(int), 9); ece = sum((bins == k).mean() * abs(yb_te[bins == k].mean() - p_ev[bins == k].mean()) for k in range(10) if (bins == k).any())
    fig, ax = plt.subplots(figsize=(4.5, 4)); ax.plot([0, 1], [0, 1], color=GREY, ls="--"); ax.plot(mean_p, frac, marker="o", color=TEAL); ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency"); style(ax); savefig(fig, "fig_exercise_calibration")
    print("ECE:", round(float(ece), 4))

saved report/fig_exercise_calibration.png
ECE: 0.0063


In [35]:
# Exercise 6 starter: cost-sensitive threshold (missed event costs 5 false alarms)
if "p_ev" in dir():
    costs = [(t, 5 * ((p_ev <= t) & (yb_te == 1)).sum() + ((p_ev > t) & (yb_te == 0)).sum()) for t in np.linspace(0.05, 0.95, 19)]
    t_best = min(costs, key=lambda x: x[1])[0]; print("cost-minimising threshold:", round(t_best, 2), "| cost:", min(c for _, c in costs))

cost-minimising threshold: 0.2 | cost: 1261


### Tự đánh giá Notebook_D (điền trước khi gửi giảng viên)

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Sáu họ mô hình, năm seed, mọi tầm | | table_rq2_summary.csv |
| Thắng naive mùa có kiểm định | | MASE, Wilcoxon |
| Tổng quát hoá theo thời gian và đơn vị | | table_rq2_rolling.csv, table_rq2_unseen_units.csv |
| Khoảng tin cậy và cảnh báo | | table_rq3_coverage_by_unit.csv, table_rq3_warning.csv |
| Đổi chế độ và ablation | | table_rq3_regime.csv, table_rq3_ablation.csv |
| SHAP nhất quán với ablation | | table_shap.csv |
| Bài tập mở rộng | | exercises_D.md |
| AI Audit Log | | ai_audit_log.csv |

## Lỗi thường gặp và cách sửa (mô hình)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Rò rỉ tương lai (điểm quá tốt) | dùng `shift(-k)` hoặc trung bình cả chuỗi khi tạo đặc trưng | chỉ dùng `LAG` và cửa sổ `PRECEDING`; kiểm tra `feat.ts` của đặc trưng luôn nhỏ hơn mục tiêu |
| Trùng dòng theo đơn vị và thời gian | nguồn có hai bản ghi cùng thời điểm | `drop_duplicates([UNIT_COL, TIME_COL])` rồi ghi vào nhật ký làm sạch |
| Lưới thời gian thiếu bước | không `resample(FREQ)` | resample và đếm bước thiếu theo đơn vị |
| Ghép nguồn phụ mất nửa dòng | lệch múi giờ hoặc lệch tần suất | đưa cả hai về UTC hoặc giờ địa phương thống nhất, dùng `merge_asof` với `tolerance` |
| DuckDB `Binder Error` | tên cột là từ khoá (`do`, `at`, `year`) hoặc trùng tên bảng | đổi tên cột, dùng dấu ngoặc kép |
| `MemoryError` | nạp toàn bộ file lớn bằng pandas | dùng `duckdb.read_csv_auto` và lọc sớm |
| Chronos không cài được | thiếu `pip install chronos-forecasting` | bỏ qua ô mô hình nền, ghi vào hạn chế |
| `ValueError: Input contains NaN` | đặc trưng trễ thiếu ở đầu chuỗi | `dropna` như Bước 4a; không điền 0 |
| Điểm quá tốt (MASE gần 0) | rò rỉ: đặc trưng chứa tương lai hoặc mục tiêu | kiểm tra ô rò rỉ ở Notebook_C, bỏ `y_diff1` nếu tính sai chiều |
| Chronos chạy quá lâu | ngữ cảnh dài, nhiều đơn vị | giảm `CTX`, số đơn vị, hoặc chạy Colab GPU |

## Checklist Notebook_D trước khi viết bài

- [ ] Bảng RQ2 summary có trung bình và độ lệch chuẩn qua 5 seed; MASE của mô hình tốt nhất dưới 1
- [ ] Hình so sánh mô hình có số in đậm trên cột, không thanh sai số; hình theo tầm
- [ ] Bảng rolling origin và bảng unseen units (RQ2 sâu)
- [ ] Bao phủ conformal theo đơn vị; cảnh báo sự kiện với recall và precision; đổi chế độ; ablation nguồn phụ (RQ3)
- [ ] SHAP top-5 và ổn định seed nhất quán với ablation
- [ ] Bảng chẩn đoán điền đủ số; test cases qua
- [ ] AI Audit Log có ít nhất 6 mục cho tuần 2 và 3, ít nhất một mục `hallucination=True`

## Mẫu viết phần Results và Discussion (điền số từ các bảng)

**RQ2.** On horizon {H0}, {best} achieves MAE {mae} (MASE {mase}), {gain}% below the seasonal naive baseline and {p} in a paired Wilcoxon test against {runner} (Table RQ2). The ranking holds across {k} rolling origins (spread {spread}) and the model keeps {pct}% of its accuracy on units never seen in training (Table unseen units). Zero-shot foundation models reach MAE {fm}, {comparison} the covariate-aware global model.

**RQ3.** Split conformal intervals cover {cov}% of test values with half-width {w}; {n} units fall below 0.85 coverage. A warning at {factor} times the event threshold reaches recall {rec} and precision {prec}; the conformal classifier holds the false-alarm rate at {fa}% with recall {rec2}. Removing the secondary source raises MAE by {abl}%, and monthly retraining reduces the post-shift error from {e1} to {e2}.

**Discussion.** Two caveats bound the claims: {caveat 1 from the diagnosis table} and {caveat 2}. Practically, {who} can use the {artifact} to {decision}, and the next step is {next}.